In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras import regularizers
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Load the Iris dataset from a common machine learning library (scikit-learn).
# The data is a set of 150 samples with 4 features (sepal length, sepal width, etc.).
# There are 3 target classes: Iris Setosa, Iris Versicolor, and Iris Virginica.
from sklearn.datasets import load_iris
iris = load_iris()
X = iris.data
y = iris.target

In [3]:
# Split the data into training and testing sets. This is crucial to evaluate
# the model on data it has never seen before. We'll use 20% of the data for testing.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
# Standardize the features. This scales all feature values to have a mean of 0
# and a standard deviation of 1, which helps neural networks learn more effectively.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# For multi-class classification, our integer labels (0, 1, 2) must be converted
# to a "one-hot encoded" format.
# Example: 2 becomes [0., 0., 1.]
num_classes = 3
y_train_one_hot = to_categorical(y_train, num_classes=num_classes)
y_test_one_hot = to_categorical(y_test, num_classes=num_classes)

In [7]:
# Define the number of input features (4 for the Iris dataset).
num_features = X_train.shape[1]

model = Sequential([
    # Input Layer and First Hidden Layer
    # We add L2 regularization (L2(0.001)) to penalize large weights, which helps prevent overfitting.
    Dense(64, activation='relu', input_shape=(num_features,), kernel_regularizer=regularizers.l2(0.001)),

    # Batch Normalization Layer (New!)
    # This layer normalizes the activations of the previous layer, which can speed up
    # training and make the network less sensitive to initial weights.
    BatchNormalization(),

    # Dropout Layer (New!)
    # We add Dropout with a rate of 0.3, meaning 30% of the neurons will be randomly
    # "turned off" during each training step. This forces the network to learn more
    # robust features and prevents it from over-relying on specific neurons.
    Dropout(0.3),

    # Second Hidden Layer
    Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),

    # Batch Normalization Layer
    BatchNormalization(),

    # Dropout Layer
    Dropout(0.3),

    # Output Layer
    # The output layer has 3 neurons (one for each class). The 'softmax' activation
    # converts the outputs into a probability distribution that sums to 1.
    Dense(num_classes, activation='softmax')
])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,187 (20.26 KB)

 Trainable params: 4,931 (19.26 KB)

 Non-trainable params: 256 (1.00 KB)

In [8]:
# --- Step 3: Compile the Model ---
# Use 'adam' optimizer and 'categorical_crossentropy' for multi-class classification.
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [9]:
# --- Step 4: Train the Model ---
# We train the model for 50 epochs with a batch size of 16.
# We also use 20% of the training data as a validation set to monitor performance.
history = model.fit(X_train, y_train_one_hot,
                    epochs=50,
                    batch_size=16,
                    validation_split=0.2,
                    verbose=1)

Epoch 1/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 72ms/step - accuracy: 0.3396 - loss: 1.6947 - val_accuracy: 0.4583 - val_loss: 1.2044
Epoch 2/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5869 - loss: 1.0894 - val_accuracy: 0.5000 - val_loss: 1.1114
Epoch 3/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7339 - loss: 0.7479 - val_accuracy: 0.7917 - val_loss: 1.0415
Epoch 4/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6836 - loss: 0.6766 - val_accuracy: 0.9167 - val_loss: 0.9835
Epoch 5/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.8176 - loss: 0.4446 - val_accuracy: 0.9167 - val_loss: 0.9381
Epoch 6/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.8379 - loss: 0.4491 - val_accuracy: 0.9167 - val_loss: 0.9047
Epoch 7/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8722 - loss: 0.4525 - val_accuracy: 0.9167 - val_loss: 0.8694
Epoch 8/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8943 - loss: 0.3666 - val_accuracy: 0.9167 - val_loss: 0.8341


In [10]:
# Evaluate the model's performance on the unseen test data.
loss, accuracy = model.evaluate(X_test, y_test_one_hot, verbose=2)
print(f"\nTest Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# --- Additional Evaluation Metrics (Beyond a single number) ---
print("\n--- Detailed Classification Report ---")
# Predict the class probabilities for the test data.
y_pred_probs = model.predict(X_test)
# Convert the probabilities to class predictions (the class with the highest probability).
y_pred = np.argmax(y_pred_probs, axis=1)

# Generate a classification report, which provides precision, recall, and f1-score
# for each class, as well as an overall average.
print(classification_report(y_test, y_pred, target_names=iris.target_names))

1/1 - 0s - 98ms/step - accuracy: 0.9667 - loss: 0.1480

Test Loss: 0.1480
Test Accuracy: 0.9667

--- Detailed Classification Report ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 172ms/step
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.89      0.94         9
   virginica       0.92      1.00      0.96        11

    accuracy                           0.97        30
   macro avg       0.97      0.96      0.97        30
weighted avg       0.97      0.97      0.97        30

